# Load Battery Dataset

In [1]:
import os
os.chdir('..')
import json
import pandas as pd
import glob

files = glob.glob('data/raw/FastCharge*.json')
print(f"Found {len(files)} files")

Found 140 files


In [ ]:
cells = []

for i, file in enumerate(files):
    try:
        print(f"Processing {i}: {file}")
        
        with open(file) as f:
            data = json.load(f)
        
        summary = pd.DataFrame(data['summary'])
        summary = summary[summary['cycle_index'] >= 1]
        
        if len(summary) == 0:
            continue
        
        initial_cap = summary['discharge_capacity'].iloc[0]
        threshold = 0.88 * initial_cap
        below_threshold = summary[summary['discharge_capacity'] < threshold]
        
        if len(below_threshold) > 0:
            eol = below_threshold.iloc[0]['cycle_index']
        else:
            eol = len(summary)
        
        cells.append({
            'cell_id': data['barcode'],
            'EOL': eol,
            'initial_capacity': initial_cap
        })
    except Exception as e:
        print(f"ERROR on {file}: {e}")
        break

print(f"Processed {len(cells)} cells")

In [ ]:
dataset = pd.DataFrame(cells)
dataset.to_csv('data/battery_dataset.csv', index=False)
print("Saved to data/battery_dataset.csv")
print(dataset.head())